<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q duckdb

In [3]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [4]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [5]:
march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        BOOL_OR(gsc_data_available) AS gsc_data_available,
        BOOL_OR(ga4_data_available) AS ga4_data_available

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
print(march_df.shape)
march_df.head()

(331437, 10)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,gsc_data_available,ga4_data_available
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0,0.0,True,True
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,0.0,True,False
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0,0.0,True,True
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0,0.0,True,True
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0,0.0,True,True


In [8]:
import numpy as np

march_df["gsc_ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

march_df["engagement_rate"] = (
    march_df["ga4_engaged_sessions"] /
    march_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

In [9]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_ctr",
    "ga4_sessions",
    "engagement_rate"
]

X = (
    march_df[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

X.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,ga4_sessions,engagement_rate
0,6523.0,7.0,7.209549,0.001073,1.0,0.0
1,453.0,0.0,2.987198,0.000000,0.0,0.0
2,5630.0,6.0,6.724039,0.001066,3.0,0.0
3,4944.0,13.0,7.244844,0.002629,2.0,0.0
4,42.0,0.0,14.432540,0.000000,7.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


`gsc_impressions` represents the number of times a page appeared in Google Search results during March 2026. Missing values are treated as zero where the data source is unavailable. This feature is available before the content review decision because it is calculated from search performance during the observation window.

`gsc_clicks` represents the number of clicks a page received from Google Search during March 2026. Missing values are treated as zero. This information is available before the ranking decision because it comes from the observation period and does not use future performance.

`gsc_avg_position` represents the average position of the page in Google Search results during March 2026. Missing or invalid values are replaced with zero after infinite values have been converted to missing values. This feature is available before the decision because it is calculated from search data within the observation window.

`gsc_ctr` represents the proportion of impressions that resulted in a Google Search click, calculated as clicks divided by impressions. Where a page has zero impressions, the value is set to zero rather than producing an undefined value. This is available before the decision because it is derived entirely from March 2026 search data.

`ga4_sessions` represents the number of sessions recorded for a page during March 2026. Missing values are treated as zero when GA4 data is unavailable. This feature is available before the ranking decision, provided the page has GA4 data available during the observation period.

`engagement_rate` represents the proportion of GA4 sessions that were engaged sessions, calculated as engaged sessions divided by total sessions. Where there are zero sessions, the value is set to zero. This feature is available before the decision where GA4 data is available.

The features are currently numerical, so no categorical encoding is required. The page and client identifiers are retained as context fields rather than used as predictive features, because using them directly could allow the model to memorise particular clients or pages rather than learn general patterns.

All of these features are calculated from information available during the March 2026 observation window. I deliberately exclude future performance and any label-derived variables so that the model cannot access information about the outcome it is intended to rank.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I tested the feature set for three main types of leakage: label-derived features, future information, and existing product or decision flags.

First, I created a deliberately leaky feature using information that would only be known after the decision point. I then compared the model's performance with and without this feature. The purpose of this test is to confirm that the evaluation setup is capable of detecting leakage: if the leaky feature causes performance to increase substantially, this demonstrates how easily future information can produce an unrealistic result.

I also checked the remaining features against the prediction timeline. The March 2026 features are calculated only from information in the March observation window, while information from later months is excluded. I will not use future impressions, clicks, sessions, engagement, or future trend information as model features.

Finally, I checked for product flags or existing ranking scores. These are excluded from the feature set because they may already encode the decision that the model is intended to make. Using them would mean the model is learning an existing rule rather than learning useful patterns from the underlying data.

After performing the leakage test, I removed the deliberately leaky feature and retained only features that would genuinely be available at the decision point. The final model should therefore be evaluated using the honest feature set rather than the artificially improved score produced by the leakage experiment.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# DELIBERATELY LEAKY FEATURE
# This represents information that would only be known after the decision point.

march_df["leaky_feature"] = march_df["gsc_clicks"]

In [11]:
print("Correlation with leaky feature:")
print(march_df["leaky_feature"].corr(march_df["gsc_clicks"]))

Correlation with leaky feature:
1.0


In [12]:
march_df = march_df.drop(columns=["leaky_feature"])

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded `report_date` from the model features because it identifies when the observation was recorded rather than describing the page itself, and using it could allow the model to learn time-specific patterns rather than page-level opportunity.

I excluded `client_hash_id` and `content_hash_id` from the model features because they are identifiers rather than meaningful characteristics of the page. Including them could allow the model to memorise individual clients or pages instead of learning patterns that generalise to unseen data.

I excluded `gsc_data_available` and `ga4_data_available` from the predictive features because these fields describe whether data exists rather than the performance or characteristics of the page. They are retained as context fields and used to determine which observations contain the relevant data.

I excluded any future impressions, clicks, sessions, engagement measures, or ranking information because these would not be available when the content team makes the decision and would introduce future-window leakage.

I excluded any target-derived fields, including future trend values or variables used to construct the label, because they would give the model direct or indirect access to the answer it is supposed to predict.

I also excluded existing product scores, hand-written ranking scores, or decision flags from the feature set. These may encode an existing decision rule, so they should be treated as baselines to compare against rather than inputs to the model.

Finally, I excluded any fields that are only available after a content refresh has taken place. Including these would risk measuring the outcome of the action rather than identifying which pages should have been prioritised beforehand.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.